In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("../src")

In [ ]:
from neoolaf.core.pipeline import Pipeline
from neoolaf.core.pipeline_state import PipelineState
from neoolaf.core.runner import Runner

from neoolaf.domain.documents import Document
from neoolaf.domain.user_guidance import UserGuidance

from neoolaf.preprocessing.pdf_parsing import extract_text_from_pdf
from neoolaf.resources.llm_backends.ollama_backend import OllamaBackend

from neoolaf.layers.layer00_preprocessing.component import PreprocessingLayer
from neoolaf.layers.layer01_linguistic_expression_extraction.component import (
    LinguisticExpressionExtractionLayer,
)
from neoolaf.resources.translation.deep_translator_backend import DeepTranslatorBackend
from neoolaf.layers.layer00_preprocessing.component import PreprocessingLayer

pdf_path = "../data/XQuality/Textual/Chapitre_8_Alarmes_et_messages.pdf"
model_name = "gemma2:27b-instruct-q4_K_M"

raw_text = extract_text_from_pdf(pdf_path)

document = Document(
    doc_id="pdf_test_001",
    source_path=pdf_path,
    raw_text=raw_text,
)

guidance = UserGuidance(
    domain_focus="industrial maintenance and causal failure chains",
    abstraction_level="Treat machine types, failure types, and symptom types as concepts; treat document-specific occurrences as individuals.",
    priority_relations=["causal", "part-of", "affects", "observed-by", "temporal"],
    population_policy="Promote a candidate to concept only if it appears recurrently across documents.",
    event_modeling_preference="Treat failures, alarms, degradations, and shutdowns as events/states rather than simple entities.",
)

state = PipelineState(
    document=document,
    llm_model=model_name,
    user_guidance=guidance,
)

ollama = OllamaBackend()

translator = DeepTranslatorBackend(max_chars=4000)

pipeline = Pipeline(
    layers=[
        PreprocessingLayer(
            chunk_size=1500,
            overlap=200,
            translate=True,          # enable translation
            translator=translator,   # use free translator
            source_language="fr",    # for example
            target_language="en",
        ),
        LinguisticExpressionExtractionLayer(
            ollama_backend=ollama,
            max_chunks=2,
            temperature=0.0,
        ),
    ]
)


runner = Runner(pipeline)
final_state = runner.run(state)

final_state.logs

['[layer00_preprocessing] started',
 '[layer00_preprocessing] translation applied',
 '[layer00_preprocessing] produced 27 chunks',
 '[layer00_preprocessing] finished',
 '[layer01_linguistic_expression_extraction] started',
 '[layer01_linguistic_expression_extraction] extracted 15 unique expressions',
 '[layer01_linguistic_expression_extraction] finished']

In [4]:
len(final_state.document.chunks)

27

In [5]:
for expr in final_state.linguistic_expressions[:10]:
    print(expr)

LinguisticExpression(expr_id='expr_00000', text='alarms', label='failure symptom', justification='', evidence=[Evidence(chunk_id='chunk_0000', chunk_start_char=40, chunk_end_char=46, doc_start_char=40, doc_end_char=46, snippet='SPRINT32linear-FA-MP-v0.1-en Page 8-4 8 Alarms and messages 8.1 Introduction Alarms and messages are divided into two main cat')], confidence=None)
LinguisticExpression(expr_id='expr_00001', text='messages', label='failure symptom', justification='', evidence=[Evidence(chunk_id='chunk_0000', chunk_start_char=51, chunk_end_char=59, doc_start_char=51, doc_end_char=59, snippet='SPRINT32linear-FA-MP-v0.1-en Page 8-4 8 Alarms and messages 8.1 Introduction Alarms and messages are divided into two main categories: 1 al')], confidence=None)
LinguisticExpression(expr_id='expr_00002', text='immediate and controlled stop', label='operational state', justification='', evidence=[Evidence(chunk_id='chunk_0000', chunk_start_char=554, chunk_end_char=583, doc_start_char=554, doc

In [ ]:
# ---------------------------------------------------------
# Path setup
# ---------------------------------------------------------
import sys
from pathlib import Path
from pprint import pprint

PROJECT_ROOT = Path("..").resolve()
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

# ---------------------------------------------------------
# NeoOLAF imports
# ---------------------------------------------------------
from neoolaf.core.pipeline import Pipeline
from neoolaf.core.pipeline_state import PipelineState
from neoolaf.core.runner import Runner

from neoolaf.domain.documents import Document
from neoolaf.domain.user_guidance import UserGuidance

from neoolaf.preprocessing.pdf_parsing import extract_text_from_pdf

from neoolaf.resources.llm_backends.ollama_backend import OllamaBackend
from neoolaf.resources.translation.deep_translator_backend import DeepTranslatorBackend

from neoolaf.resources.knowledge_sources.wordnet_source import WordNetSource
from neoolaf.resources.knowledge_sources.wikipedia_source import WikipediaSource
from neoolaf.resources.knowledge_sources.wikidata_source import WikidataSource
from neoolaf.resources.knowledge_sources.web_search_source import WebSearchSource

from neoolaf.layers.layer00_preprocessing.component import PreprocessingLayer
from neoolaf.layers.layer01_linguistic_expression_extraction.component import (
    LinguisticExpressionExtractionLayer,
)
from neoolaf.layers.layer02_candidate_enrichment.component import (
    CandidateEnrichmentLayer,
)

# ---------------------------------------------------------
# Config
# ---------------------------------------------------------
PDF_PATH = "../data/XQuality/Textual/Chapitre_8_Alarmes_et_messages.pdf"
MODEL_NAME = "gemma2:27b-instruct-q4_K_M"

# Debug limits
MAX_CHUNKS = 2
MAX_EXPRESSIONS = 5

# ---------------------------------------------------------
# Load document
# ---------------------------------------------------------
raw_text = extract_text_from_pdf(str(PDF_PATH))

document = Document(
    doc_id="pdf_test_001",
    source_path=str(PDF_PATH),
    raw_text=raw_text,
)

# ---------------------------------------------------------
# Optional semantic guidance
# ---------------------------------------------------------
guidance = UserGuidance(
    domain_focus="industrial maintenance and causal failure chains",
    abstraction_level=(
        "Treat machine types, failure types, and symptom types as concepts; "
        "treat document-specific occurrences as individuals."
    ),
    priority_relations=["causal", "part-of", "affects", "observed-by", "temporal"],
    population_policy="Promote a candidate to concept only if it appears recurrently across documents.",
    event_modeling_preference="Treat failures, alarms, degradations, and shutdowns as events/states rather than simple entities.",
)

# ---------------------------------------------------------
# State and backends
# ---------------------------------------------------------
state = PipelineState(
    document=document,
    llm_model=MODEL_NAME,
    user_guidance=guidance,
)

ollama = OllamaBackend()
translator = DeepTranslatorBackend(max_chars=4000)

wordnet_source = WordNetSource()
wikipedia_source = WikipediaSource(language="en")
wikidata_source = WikidataSource(language="en")
web_search_source = WebSearchSource()

# ---------------------------------------------------------
# Pipeline
# ---------------------------------------------------------
pipeline = Pipeline(
    layers=[
        PreprocessingLayer(
            chunk_size=1500,
            overlap=200,
            translate=False,
            translator=translator,
        ),
        LinguisticExpressionExtractionLayer(
            ollama_backend=ollama,
            max_chunks=MAX_CHUNKS,
            temperature=0.0,
        ),
        CandidateEnrichmentLayer(
            ollama_backend=ollama,
            wikipedia_source=wikipedia_source,
            wikidata_source=wikidata_source,
            wordnet_source=wordnet_source,
            web_search_source=web_search_source,
            max_expressions=MAX_EXPRESSIONS,
            use_web_search=True,
        ),
    ]
)

runner = Runner(pipeline)
final_state = runner.run(state)

# ---------------------------------------------------------
# Inspect one enriched expression
# ---------------------------------------------------------
item = final_state.enriched_expressions[0]

print("Base expression:", item.base_expression.text)
print("Aliases:", item.aliases)
print("Alias sources:")
pprint(item.alias_sources)

print("\nSynonyms:", item.synonyms)
print("Synonym sources:")
pprint(item.synonym_sources)

print("\nLexical variants:", item.lexical_variants)
print("Lexical variant sources:")
pprint(item.lexical_variant_sources)

print("\nDefinition:", item.definition)
print("Ontology hints:", item.ontology_hints)

print("\nEvidence sources:")
for ev in item.enrichment_evidence:
    print("-", ev.source, "|", ev.reference)

Base expression: alarms
Aliases: ['alarm', 'dismay', 'consternation', 'warning device', 'alarm system', 'alert', 'warning signal', 'alarum', 'alarm clock', 'appal', 'appall', 'horrify', 'Alarm device']
Alias sources:
{'Alarm device': ['wikipedia', 'llm'],
 'alarm': ['wordnet'],
 'alarm clock': ['wordnet'],
 'alarm system': ['wordnet'],
 'alarum': ['wordnet'],
 'alert': ['wordnet'],
 'appal': ['wordnet'],
 'appall': ['wordnet'],
 'consternation': ['wordnet'],
 'dismay': ['wordnet'],
 'horrify': ['wordnet'],
 'warning device': ['wordnet'],
 'warning signal': ['wordnet']}

Synonyms: ['alarm', 'dismay', 'consternation', 'warning device', 'alarm system', 'alert', 'warning signal', 'alarum', 'alarm clock', 'appal', 'appall', 'horrify']
Synonym sources:
{'alarm': ['wordnet'],
 'alarm clock': ['wordnet'],
 'alarm system': ['wordnet'],
 'alarum': ['wordnet'],
 'alert': ['wordnet', 'llm'],
 'appal': ['wordnet'],
 'appall': ['wordnet'],
 'consternation': ['wordnet'],
 'dismay': ['wordnet'],
 'hor

In [26]:
final_state.enriched_expressions

[EnrichedExpression(base_expression=LinguisticExpression(expr_id='expr_00000', text='alarms', label='failure symptom', justification='', evidence=[Evidence(chunk_id='chunk_0000', chunk_start_char=-1, chunk_end_char=-1, doc_start_char=-1, doc_end_char=-1, snippet='SPRINT32linear-FA-MP-v0.1-fr Page 8-4 8 Alarmes et messages 8.1 Introduction Les alarmes et les messages sont divisés en deux catégories principales : 1 les alarmes et messages de la CNC, pour lesquels nous renvoyons au ma- nuel d’entretien de la CNC Fanuc série ........ 2 les alarmes et messages de')], confidence=None), aliases=['alarm', 'alert'], synonyms=['warning signal', 'warning device'], lexical_variants=[], definition='a signal indicating a problem or condition requiring urgent attention.', ontology_hints=['failure symptom', 'observed-by: sensors'], enrichment_evidence=[EnrichmentEvidence(source='wordnet', content='fear resulting from the awareness of danger', reference=None), EnrichmentEvidence(source='wordnet', conte

In [3]:
PDF_PATH = pdf_path = "../data/XQuality/Textual/Chapitre_8_Alarmes_et_messages.pdf"
MODEL_NAME = model_name = "gemma2:27b-instruct-q4_K_M"

In [ ]:
# ---------------------------------------------------------
# Quick test after patching Layer 1 + Layer 3
# ---------------------------------------------------------
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from neoolaf.core.pipeline import Pipeline
from neoolaf.core.pipeline_state import PipelineState
from neoolaf.core.runner import Runner

from neoolaf.domain.documents import Document
from neoolaf.domain.user_guidance import UserGuidance

from neoolaf.preprocessing.pdf_parsing import extract_text_from_pdf

from neoolaf.resources.llm_backends.ollama_backend import OllamaBackend
from neoolaf.resources.translation.deep_translator_backend import DeepTranslatorBackend

from neoolaf.resources.knowledge_sources.wordnet_source import WordNetSource
from neoolaf.resources.knowledge_sources.wikipedia_source import WikipediaSource
from neoolaf.resources.knowledge_sources.wikidata_source import WikidataSource
from neoolaf.resources.knowledge_sources.web_search_source import WebSearchSource

from neoolaf.layers.layer00_preprocessing.component import PreprocessingLayer
from neoolaf.layers.layer01_linguistic_expression_extraction.component import (
    LinguisticExpressionExtractionLayer,
)
from neoolaf.layers.layer02_candidate_enrichment.component import (
    CandidateEnrichmentLayer,
)
from neoolaf.layers.layer03_candidate_typing_resolution.component import (
    CandidateTypingResolutionLayer,
)


raw_text = extract_text_from_pdf(str(PDF_PATH))

document = Document(
    doc_id="pdf_relation_test",
    source_path=str(PDF_PATH),
    raw_text=raw_text,
)

guidance = UserGuidance(
    domain_focus="industrial maintenance and causal failure chains",
    abstraction_level="Treat machine types, failure types, and symptom types as concepts; treat document-specific occurrences as individuals.",
    priority_relations=["causal", "part-of", "affects", "observed-by", "temporal"],
    population_policy="Promote a candidate to concept only if it appears recurrently across documents.",
    event_modeling_preference="Treat failures, alarms, degradations, and shutdowns as events/states rather than simple entities.",
)

state = PipelineState(
    document=document,
    llm_model=MODEL_NAME,
    user_guidance=guidance,
)

ollama = OllamaBackend()
translator = DeepTranslatorBackend(max_chars=4000)

pipeline = Pipeline(
    layers=[
        PreprocessingLayer(
            chunk_size=1500,
            overlap=200,
            translate=False,
            translator=translator,
        ),
        LinguisticExpressionExtractionLayer(
            ollama_backend=ollama,
            max_chunks=2,
            temperature=0.0,
        ),
        CandidateEnrichmentLayer(
            ollama_backend=ollama,
            wikipedia_source=WikipediaSource(language="en"),
            wikidata_source=WikidataSource(language="en"),
            wordnet_source=WordNetSource(),
            web_search_source=WebSearchSource(),
            max_expressions=20,
            use_web_search=True,
        ),
        CandidateTypingResolutionLayer(
            ollama_backend=ollama,
            max_expressions=20,
            temperature=0.0,
        ),
    ]
)

runner = Runner(pipeline)
final_state = runner.run(state)

print("Entities:", len(final_state.entity_candidates))
print("Relations:", len(final_state.relation_candidates))
print("Attributes:", len(final_state.attribute_candidates))
print("Events:", len(final_state.event_candidates))

print("\nRelation candidates:")
for cand in final_state.relation_candidates[:15]:
    print("-", cand.candidate_id, "|", cand.canonical_label)

In [1]:
print(len(final_state.document.chunks))
print(len(final_state.linguistic_expressions))
print(len(final_state.enriched_expressions))
print(len(final_state.entity_candidates))
print(len(final_state.relation_candidates))
print(len(final_state.attribute_candidates))
print(len(final_state.event_candidates))

NameError: name 'final_state' is not defined